# 🤖 AI-Assisted & Practical Coding Interview Guide
### Stateful Simulations, Robot Navigation, Interactive Games & Progressive System Design

**Target Roles:** Machine Learning Engineer (MLE), AI Systems Engineer, Applied Scientist, Software Engineer

---

## 📌 Overview: Practical Implementation & AI-Assisted Coding Rounds

Across the software engineering and machine learning interview landscape, coding rounds have increasingly expanded beyond traditional algorithmic riddles. These kinds of questions get asked in companies like **Shopify**, **LinkedIn**, **Meta**, and similar tech organizations, while many other companies have also adopted comparable AI-assisted pair-programming or progressive implementation formats.

In these rounds (often conducted in local IDEs like Cursor/VS Code with Copilot or in shared CoderPad environments with AI panels), candidates are given bare-bones class structures and domain rules. Rather than testing memorization of obscure competitive programming tricks, these sessions evaluate:

1. **Extensible Architecture (Open-Closed Principle):** Can your class design cleanly absorb follow-up requirements (e.g., Undo, Jump, Eviction policies) without rewriting existing logic?
2. **State Machine Integrity:** Can you manage multi-step states (headings, obstacle sets, body queues, remaining TTL intervals) without bugs?
3. **AI Verification & Edge Cases:** AI assistants often produce code with subtle bugs (boundary off-by-ones, snake tail vacation timing, expired records in backups). Can you spot and correct them?
4. **Clean Code & Verification:** Can you write concise, functional code and immediately prove its correctness with assertions?

---

## 🗺️ Curriculum Overview

| Module | # | Problem | Core Pattern / Archetype | Concepts Tested |
|---|---|---|---|---|
| **1. Robot Navigation & Grid Simulation** | 1 | **Mars Rover / Grid Robot Command Engine** | Grid Navigation & Collision | Cardinal directions, ray-step obstacle collision, undo history |
| | 2 | **Blind-Grid Robot Room Cleaner** | Sensor Traversal & Backtracking | Sensor API, virtual coordinates, blind DFS backtracking with orientation restoration |
| | 3 | **Cleaning Robot with Cycle Detection** | Matrix Simulation & Cycle Detection | Deterministic grid reflection, state space cycle detection |
| **2. Interactive Game Engines & State Machines** | 4 | **Snake Game Simulator with Growth & Collision** | Stateful Game Simulation | deque body, sequential food ingestion, tail vacation vs head collision order |
| | 5 | **Connect Four / Tic-Tac-Toe with O(1) Win Checking** | Low-Level System Design | Vector counters (rows, cols, diagonals), O(1) evaluation, gravity drop |
| | 6 | **Battleship Fleet Coordinator** | Grid Geometry & Event Handling | Multi-deck ship geometry, hit/miss tracking, sinking validation |
| **3. Progressive Systems & Storage** | 7 | **In-Memory Key-Field DB with TTL & Backup/Restore** | Progressive 4-Level Architecture | CRUD -> Prefix Scan -> TTL interval -> Remaining TTL restore |
| | 8 | **Cloud Storage Manager with Quota & Auto-Eviction** | Resource Quotas & Auto-Eviction | User quota management, largest-first tie-broken auto-eviction, file compression |
| | 9 | **Text Editor with Undo/Redo & Find/Replace** | Buffer Manipulation & Command History | Buffer slicing, pattern replacement count, transactional Command Undo/Redo |
| **4. Enterprise Infrastructure & Scheduling** | 10 | **Multi-Account Financial Ledger with Rollbacks** | Double-Entry & Atomic Transactions | Double-entry bookkeeping, point-in-time binary search balance, atomic rollback |
| | 11 | **Sliding Window Log & Token Bucket Rate Limiter** | Traffic Shaping & Concurrency | Timestamp sliding window log, continuous refill token bucket |
| | 12 | **DAG Task Engine with Dependency Resolution** | Graph Pipelines & Orchestration | Directed acyclic graph, Kahn's topological sort, cycle detection, execution pipeline |


---
## 1. Mars Rover / Grid Robot Command Engine

### Problem Statement
Design a controller for a robot operating on an unbounded 2D Cartesian grid.
- The robot starts at coordinate `(x, y)` facing one of four cardinal directions: `'N'`, `'E'`, `'S'`, or `'W'`.
- It executes a sequence of commands passed as a space-separated string:
  - `'F'`: Move forward 1 unit in current direction.
  - `'B'`: Move backward 1 unit in current direction (maintaining heading).
  - `'L'`: Turn 90° counter-clockwise (e.g., `'N'` $\to$ `'W'`).
  - `'R'`: Turn 90° clockwise (e.g., `'N'` $\to$ `'E'`).
  - `'J<k>'`: Jump forward $k$ units in current direction (e.g., `'J3'`).
- **Obstacle Collision:** If moving forward/backward/jumping would cross or land on a known obstacle `(ox, oy)`, the robot **halts immediately before the obstacle** and completes no further steps of that movement command.
- **Undo Capability:** `undo(steps)` reverts the robot's state to what it was `steps` commands prior.

### Starter Interface
```python
class GridRobot:
    def __init__(self, x: int = 0, y: int = 0, direction: str = 'N', obstacles: set[tuple[int, int]] = None):
        pass

    def execute(self, commands: str) -> tuple[int, int, str]:
        """Executes commands; returns (x, y, direction)."""
        pass

    def undo(self, steps: int = 1) -> tuple[int, int, str]:
        """Reverts the last `steps` actions; returns (x, y, direction)."""
        pass
```

### Key Traps & AI Failure Modes
- **Obstacle Jumping:** AI assistants often only check if the final destination has an obstacle, forgetting that the robot cannot teleport through an obstacle between the start and destination! Ray-step checking is required.
- **Direction Modulo:** Modulo arithmetic with negative numbers in some languages can produce negative offsets; in Python `(idx - 1) % 4` is safe.


In [ ]:
from __future__ import annotations
from typing import List, Tuple, Set, Optional

class GridRobot:
    """Stateful 2D grid robot simulator with cardinal navigation, obstacle avoidance, and undo history."""
    DIRS = ['N', 'E', 'S', 'W']
    OFFSETS = {'N': (0, 1), 'E': (1, 0), 'S': (0, -1), 'W': (-1, 0)}

    def __init__(self, x: int = 0, y: int = 0, direction: str = 'N', obstacles: Optional[Set[Tuple[int, int]]] = None):
        self.x = x
        self.y = y
        self.direction = direction
        self.obstacles = set(obstacles) if obstacles else set()
        self.history: List[Tuple[int, int, str]] = []

    def execute(self, commands: str) -> Tuple[int, int, str]:
        self.history.append((self.x, self.y, self.direction))
        for token in commands.split():
            if token == 'L':
                idx = self.DIRS.index(self.direction)
                self.direction = self.DIRS[(idx - 1) % 4]
            elif token == 'R':
                idx = self.DIRS.index(self.direction)
                self.direction = self.DIRS[(idx + 1) % 4]
            elif token == 'F':
                self._step(1)
            elif token == 'B':
                self._step(-1)
            elif token.startswith('J'):
                dist = int(token[1:]) if len(token) > 1 else 2
                self._step(dist)
        return (self.x, self.y, self.direction)

    def _step(self, steps: int) -> None:
        """Step incrementally along line of sight to catch intermediate obstacles."""
        dx, dy = self.OFFSETS[self.direction]
        step_dir = 1 if steps > 0 else -1
        for _ in range(abs(steps)):
            nx = self.x + dx * step_dir
            ny = self.y + dy * step_dir
            if (nx, ny) in self.obstacles:
                break
            self.x, self.y = nx, ny

    def undo(self, steps: int = 1) -> Tuple[int, int, str]:
        for _ in range(min(steps, len(self.history))):
            self.x, self.y, self.direction = self.history.pop()
        return (self.x, self.y, self.direction)

# ---------------- Verification Tests ----------------
robot = GridRobot(x=0, y=0, direction='N', obstacles={(0, 2), (5, 5)})
assert robot.execute("F F") == (0, 1, 'N'), "Robot should stop at (0, 1) before obstacle at (0, 2)"
assert robot.execute("R F F") == (2, 1, 'E'), "Turn East and advance 2 units"
assert robot.undo(1) == (0, 1, 'N'), "Undo should restore state prior to 'R F F'"
assert robot.execute("L J3") == (-3, 1, 'W'), "Turn West and jump 3 units"
print("✓ Problem 1: GridRobot passed all tests.")


---
## 2. Blind-Grid Robot Room Cleaner

### Problem Statement
You control a cleaning robot placed inside an unknown room represented by an enclosed grid of cells.
- You are **not** given room dimensions or starting coordinates.
- You can interact with the room strictly via four sensory API calls on `RobotAPI`:
  - `move() -> bool`: Moves forward 1 cell. Returns `True` on success, or `False` if blocked by a wall.
  - `turn_left() -> None`: Rotates 90° counter-clockwise.
  - `turn_right() -> None`: Rotates 90° clockwise.
  - `clean() -> None`: Cleans the current cell.
- **Goal:** Design an algorithm that cleans every reachable space in the room and returns the total number of unique cells cleaned.

### Starter Interface
```python
class RobotAPI:
    def move(self) -> bool: pass
    def turn_left(self) -> None: pass
    def turn_right(self) -> None: pass
    def clean(self) -> None: pass

class RobotRoomCleaner:
    def clean_room(self, robot: RobotAPI) -> int:
        """Cleans all reachable spaces and returns count of unique cleaned cells."""
        pass
```

### Key Traps & AI Failure Modes
- **Backtracking State Restoration:** When backtracking in recursion, returning to the previous cell is not enough—you must also **restore the robot's physical heading** so higher-level call frames preserve their directional invariant!
- **Virtual Coordinates:** Start at virtual `(0, 0)` with heading `0` (Up). Maintain visited cells in a hash set relative to `(0, 0)`.


In [ ]:
from __future__ import annotations
from typing import Set, Tuple, List

class MockRobot:
    """Mock environment representing an enclosed room with walls."""
    DIRS = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # Up, Right, Down, Left

    def __init__(self, room: List[List[int]], start_r: int, start_c: int):
        self.room = room
        self.r = start_r
        self.c = start_c
        self.d = 0  # 0: Up, 1: Right, 2: Down, 3: Left
        self.cleaned: Set[Tuple[int, int]] = set()

    def move(self) -> bool:
        dr, dc = self.DIRS[self.d]
        nr, nc = self.r + dr, self.c + dc
        if 0 <= nr < len(self.room) and 0 <= nc < len(self.room[0]) and self.room[nr][nc] == 1:
            self.r, self.c = nr, nc
            return True
        return False

    def turn_left(self) -> None:
        self.d = (self.d - 1) % 4

    def turn_right(self) -> None:
        self.d = (self.d + 1) % 4

    def clean(self) -> None:
        self.cleaned.add((self.r, self.c))

class RobotRoomCleaner:
    DIRS = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # Relative: Up, Right, Down, Left

    def clean_room(self, robot: MockRobot) -> int:
        visited: Set[Tuple[int, int]] = set()

        def go_back():
            """Physically move 1 step backward and restore original orientation."""
            robot.turn_right()
            robot.turn_right()
            robot.move()
            robot.turn_right()
            robot.turn_right()

        def backtrack(r: int, c: int, d: int):
            visited.add((r, c))
            robot.clean()

            # Explore 4 directions clockwise starting from current heading
            for i in range(4):
                new_d = (d + i) % 4
                dr, dc = self.DIRS[new_d]
                nr, nc = r + dr, c + dc

                if (nr, nc) not in visited:
                    if robot.move():
                        backtrack(nr, nc, new_d)
                        go_back()
                robot.turn_right()

        backtrack(0, 0, 0)
        return len(visited)

# ---------------- Verification Tests ----------------
room_map = [
    [1, 1, 1, 0],
    [1, 0, 1, 1],
    [1, 1, 1, 1]
]
mock_robot = MockRobot(room_map, start_r=0, start_c=0)
cleaner = RobotRoomCleaner()

cleaned_count = cleaner.clean_room(mock_robot)
expected_reachable = sum(row.count(1) for row in room_map)

assert cleaned_count == expected_reachable, f"Expected {expected_reachable}, got {cleaned_count}"
assert len(mock_robot.cleaned) == expected_reachable
print(f"✓ Problem 2: RobotRoomCleaner cleaned all {cleaned_count} reachable cells.")


---
## 3. Autonomous Cleaning Robot with Cycle Detection

### Problem Statement
An autonomous cleaning robot is placed in an $R \times C$ room where `0` represents cleanable empty space and `1` represents an obstacle.
- The robot starts at cell `(0, 0)` facing **East** (Right).
- **Movement Rules:**
  1. It marks the current space as cleaned.
  2. It attempts to move straight forward in its current direction.
  3. If it hits an obstacle or room boundary, it **turns 90° clockwise** without moving and tries again.
- **Termination:** Because movement is deterministic, the robot terminates when it enters an infinite loop (it reaches an `(r, c, direction)` state it has already visited).
- **Goal:** Return the total number of **unique** spaces cleaned.

### Starter Interface
```python
class CleaningSimulator:
    def number_of_clean_spaces(self, room: list[list[int]]) -> int:
        """Returns total unique spaces cleaned before looping."""
        pass
```

### Key Traps & AI Failure Modes
- **Cleaned Cells vs State History:** AI assistants often confuse unique cleaned cells with visited loop states. You need two separate sets: `cleaned` tracks `(r, c)`, while `visited_states` tracks `(r, c, direction)`.


In [ ]:
from __future__ import annotations
from typing import List, Set, Tuple

class CleaningSimulator:
    """Simulates deterministic cleaning robot with cycle detection on an obstacle matrix."""
    DIRS = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # East (0), South (1), West (2), North (3)

    def number_of_clean_spaces(self, room: List[List[int]]) -> int:
        R, C = len(room), len(room[0])
        r, c, d = 0, 0, 0
        
        cleaned: Set[Tuple[int, int]] = set()
        visited_states: Set[Tuple[int, int, int]] = set()

        while True:
            cleaned.add((r, c))
            state = (r, c, d)
            
            if state in visited_states:
                break  # Cycle detected: deterministic loop reached
            visited_states.add(state)

            dr, dc = self.DIRS[d]
            nr, nc = r + dr, c + dc
            
            if 0 <= nr < R and 0 <= nc < C and room[nr][nc] == 0:
                r, c = nr, nc
            else:
                d = (d + 1) % 4  # Turn 90° clockwise

        return len(cleaned)

# ---------------- Verification Tests ----------------
sim = CleaningSimulator()
test_room = [
    [0, 0, 0],
    [1, 1, 0],
    [0, 0, 0]
]
cleaned_cells = sim.number_of_clean_spaces(test_room)
assert cleaned_cells == 7, f"Expected 7 cleaned cells, got {cleaned_cells}"
print(f"✓ Problem 3: CleaningSimulator successfully detected cycle with {cleaned_cells} unique spaces cleaned.")


---
## 4. Snake Game Simulator with Growth & Collision

### Problem Statement
Design a stateful Snake Game running on a grid of size `width` $\times$ `height`.
- The snake starts at coordinate `(0, 0)` with initial length 1.
- Food coordinates appear sequentially from a predefined list `food = [[r1, c1], [r2, c2], ...]`.
- `move(direction: str) -> int`:
  - Direction is `'U'` (Up), `'D'` (Down), `'L'` (Left), or `'R'` (Right).
  - Returns the **current score** (equal to number of food items consumed so far), or `-1` if the snake collides with a boundary or its own body.
  - If the new head coordinate matches the current food item:
    - The snake eats the food, grows by 1 (its tail does **not** move), and advances to the next food item.
  - If no food is eaten:
    - The tail moves forward (vacates its previous cell) as the head advances.

### Starter Interface
```python
class SnakeGame:
    def __init__(self, width: int, height: int, food: list[list[int]]):
        pass

    def move(self, direction: str) -> int:
        """Advances snake by 1 step. Returns current score or -1 on collision."""
        pass
```

### Key Traps & AI Failure Modes
- **The Tail Vacation Rule:** If the snake's head moves into the cell currently occupied by its tail on a normal non-eating step, **this is NOT a collision** because the tail moves forward simultaneously!
- **Data Structures:** Using a pure list for the body causes $\mathcal{O}(L)$ membership checks. Combining `collections.deque` with a `set` yields $\mathcal{O}(1)$ moves.


In [ ]:
from __future__ import annotations
from collections import deque
from typing import List, Tuple, Set

class SnakeGame:
    """Snake game engine with O(1) step execution via double-ended queue + hash set."""
    DIR_OFFSETS = {'U': (-1, 0), 'D': (1, 0), 'L': (0, -1), 'R': (0, 1)}

    def __init__(self, width: int, height: int, food: List[List[int]]):
        self.width = width
        self.height = height
        self.food = food
        self.food_idx = 0
        self.score = 0
        
        self.body: deque[Tuple[int, int]] = deque([(0, 0)])
        self.body_set: Set[Tuple[int, int]] = {(0, 0)}

    def move(self, direction: str) -> int:
        dr, dc = self.DIR_OFFSETS[direction]
        head_r, head_c = self.body[0]
        nr, nc = head_r + dr, head_c + dc

        # 1. Boundary check
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1

        # 2. Check food consumption
        is_food = (self.food_idx < len(self.food) and [nr, nc] == self.food[self.food_idx])

        # 3. Vacate tail before collision check if no food eaten
        tail = self.body[-1]
        if not is_food:
            self.body.pop()
            self.body_set.remove(tail)

        # 4. Check body collision
        if (nr, nc) in self.body_set:
            return -1

        # 5. Place new head
        self.body.appendleft((nr, nc))
        self.body_set.add((nr, nc))

        # 6. Update food and score
        if is_food:
            self.food_idx += 1
            self.score += 1

        return self.score

# ---------------- Verification Tests ----------------
game = SnakeGame(width=3, height=2, food=[[1, 2], [0, 1]])

assert game.move('R') == 0, "Move to (0, 1) - no food"
assert game.move('D') == 0, "Move to (1, 1) - no food"
assert game.move('R') == 1, "Move to (1, 2) - consumed food 1, score = 1"
assert game.move('U') == 1, "Move to (0, 2) - no food"
assert game.move('L') == 2, "Move to (0, 1) - consumed food 2, score = 2"
assert game.move('U') == -1, "Out of bounds collision -> Game Over (-1)"

print("✓ Problem 4: SnakeGame simulation verified.")


---
## 5. Connect Four / Tic-Tac-Toe Game Engine with $\mathcal{O}(1)$ Win Checking

### Problem Statement
Design an $N \times N$ board game engine for 2 players.
- `make_move(row: int, col: int, player: int) -> int`:
  - Records a move by `player` (1 or 2) at coordinate `(row, col)`.
  - Determines if the current move results in a win **in strictly $\mathcal{O}(1)$ time** without scanning the $N \times N$ board.
  - Returns:
    - `0` if no player has won yet.
    - `1` if Player 1 wins.
    - `2` if Player 2 wins.
- A player wins when they successfully place $N$ tokens in any horizontal row, vertical column, main diagonal, or anti-diagonal.

### Starter Interface
```python
class ConnectEngine:
    def __init__(self, n: int = 3):
        pass

    def make_move(self, row: int, col: int, player: int) -> int:
        """Places token; returns winner (1 or 2) or 0 if game continues."""
        pass
```

### Key Traps & AI Failure Modes
- **Full Board Scan Trap:** Scanning the board after each move takes $\mathcal{O}(N)$ or $\mathcal{O}(N^2)$ time. In production and interviews, interviewers expect **$\mathcal{O}(1)$ amortized tracking** using count vectors.
- **Anti-Diagonal Math:** The anti-diagonal condition is strictly `row + col == n - 1`.


In [ ]:
from __future__ import annotations
from collections import defaultdict

class ConnectEngine:
    """O(1) win-checking game engine using linear accumulator arrays per player."""
    def __init__(self, n: int = 3):
        self.n = n
        self.rows = defaultdict(lambda: defaultdict(int))
        self.cols = defaultdict(lambda: defaultdict(int))
        self.diag = defaultdict(int)
        self.anti_diag = defaultdict(int)
        self.winner = 0

    def make_move(self, row: int, col: int, player: int) -> int:
        if self.winner != 0:
            return self.winner

        self.rows[player][row] += 1
        self.cols[player][col] += 1
        
        if row == col:
            self.diag[player] += 1
        if row + col == self.n - 1:
            self.anti_diag[player] += 1

        if (self.rows[player][row] == self.n or
            self.cols[player][col] == self.n or
            self.diag[player] == self.n or
            self.anti_diag[player] == self.n):
            self.winner = player
            return player

        return 0

# ---------------- Verification Tests ----------------
game = ConnectEngine(n=3)

assert game.make_move(0, 0, player=1) == 0
assert game.make_move(0, 2, player=2) == 0
assert game.make_move(1, 1, player=1) == 0
assert game.make_move(1, 0, player=2) == 0
assert game.make_move(2, 2, player=1) == 1, "Player 1 should win on main diagonal in O(1)"

print("✓ Problem 5: ConnectEngine O(1) win verification passed.")


---
## 6. Battleship Fleet Coordinator

### Problem Statement
Build the coordinator for a two-player naval combat Battleship game on a grid of size `size \times size`.
- `place_ship(player: int, ship_name: str, coords: list[tuple[int, int]]) -> bool`:
  - Places a named ship for `player` (1 or 2).
  - Rejects if any coordinate is out of bounds or overlaps with an existing ship belonging to that player.
- `fire(player: int, target: tuple[int, int]) -> str`:
  - `player` fires at the opponent's coordinates.
  - Returns:
    - `"Already Targeted"` if this player previously fired at this cell.
    - `"Miss"` if the target contains no ship.
    - `"Hit"` if a ship coordinate was struck.
    - `"Sunk <ship_name>"` if all cells of the hit ship are now struck.
    - `"Victory"` if the last remaining opponent ship has been sunk.

### Starter Interface
```python
class BattleshipGame:
    def __init__(self, size: int = 10):
        pass

    def place_ship(self, player: int, ship_name: str, coords: list[tuple[int, int]]) -> bool:
        pass

    def fire(self, player: int, target: tuple[int, int]) -> str:
        """Fires shot; returns 'Miss', 'Hit', 'Sunk <name>', or 'Victory'."""
        pass
```


In [ ]:
from __future__ import annotations
from typing import Dict, Set, Tuple, List

class BattleshipGame:
    """Two-player Battleship coordinator managing fleet geometry and damage tracking."""
    def __init__(self, size: int = 10):
        self.size = size
        self.fleet: Dict[int, Dict[str, Set[Tuple[int, int]]]] = {1: {}, 2: {}}
        self.fired_shots: Dict[int, Set[Tuple[int, int]]] = {1: set(), 2: set()}

    def place_ship(self, player: int, ship_name: str, coords: List[Tuple[int, int]]) -> bool:
        c_set = set(coords)
        for r, c in c_set:
            if not (0 <= r < self.size and 0 <= c < self.size):
                return False
            for existing_ship in self.fleet[player].values():
                if (r, c) in existing_ship:
                    return False
        self.fleet[player][ship_name] = c_set
        return True

    def fire(self, player: int, target: Tuple[int, int]) -> str:
        opponent = 2 if player == 1 else 1
        if target in self.fired_shots[player]:
            return "Already Targeted"
        self.fired_shots[player].add(target)

        for name, remaining_coords in self.fleet[opponent].items():
            if target in remaining_coords:
                remaining_coords.remove(target)
                if not remaining_coords:
                    del self.fleet[opponent][name]
                    if not self.fleet[opponent]:
                        return "Victory"
                    return f"Sunk {name}"
                return "Hit"
                
        return "Miss"

# ---------------- Verification Tests ----------------
game = BattleshipGame(size=5)

assert game.place_ship(player=1, ship_name="Destroyer", coords=[(0, 0), (0, 1)]) is True
assert game.place_ship(player=2, ship_name="Patrol", coords=[(2, 2)]) is True

assert game.fire(player=1, target=(4, 4)) == "Miss"
assert game.fire(player=1, target=(4, 4)) == "Already Targeted"
assert game.fire(player=1, target=(2, 2)) == "Victory", "Sinking opponent's last ship grants Victory"

print("✓ Problem 6: BattleshipGame fleet coordination passed.")


---
## 7. In-Memory Key-Field Database with TTL & Backup/Restore

### Problem Statement
Build an in-memory database storing `(key, field) -> value` entries with 4 evolving levels:
- **Level 1 (Basic CRUD):**
  - `set(key, field, value)`: Inserts or updates a field value.
  - `get(key, field) -> str | None`: Returns value or `None` if absent.
  - `delete(key, field) -> bool`: Deletes field; returns `True` if deleted, `False` if absent.
- **Level 2 (Scanning & Formatted Output):**
  - `scan(key) -> list[str]`: Returns lexicographically sorted `["field(value)"]` strings.
  - `scan_by_prefix(key, prefix) -> list[str]`: Returns matching fields filtered by prefix.
- **Level 3 (Timestamps & Time-to-Live):**
  - `set_at_with_ttl(key, field, value, timestamp, ttl)`: Expiration occurs at `timestamp + ttl` (half-open: valid on `[set_time, set_time + ttl)`).
  - `get_at(key, field, timestamp) -> str | None`: Visible only if not expired at query time.
- **Level 4 (Backup & Restore with Recomputed Expiration):**
  - `backup(timestamp) -> int`: Snapshots all non-expired entries at `timestamp`. For each entry, records its remaining TTL: $rem\_ttl = (set\_time + ttl) - timestamp$. Returns backup ID.
  - `restore(timestamp, backup_id) -> bool`: Restores DB from backup. New expiration becomes $timestamp + rem\_ttl$. Already-expired entries are omitted.

### Starter Interface
```python
class InMemoryDB:
    def set(self, key: str, field: str, value: str) -> None: pass
    def get(self, key: str, field: str) -> str | None: pass
    def delete(self, key: str, field: str) -> bool: pass
    def scan(self, key: str) -> list[str]: pass
    def scan_by_prefix(self, key: str, prefix: str) -> list[str]: pass
    def set_at_with_ttl(self, key: str, field: str, value: str, timestamp: int, ttl: int) -> None: pass
    def get_at(self, key: str, field: str, timestamp: int) -> str | None: pass
    def backup(self, timestamp: int) -> int: pass
    def restore(self, timestamp: int, backup_id: int) -> bool: pass
```

### Key Traps & AI Failure Modes
- **Backup TTL Recomputation:** When restoring at time $T_{restore}$, resetting TTL to original $ttl$ is wrong. The remaining TTL must be recomputed: $T_{new\_expiry} = T_{restore} + rem\_ttl$.


In [ ]:
from __future__ import annotations
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

class InMemoryDB:
    """4-Level In-Memory Database with CRUD, Prefix Scan, TTL, and Recomputed Backup/Restore."""
    def __init__(self):
        self.db: Dict[str, Dict[str, Tuple[str, int, Optional[int]]]] = defaultdict(dict)
        self.backups: Dict[int, Dict[str, Dict[str, Tuple[str, Optional[int]]]]] = {}
        self.backup_counter = 0

    # Level 1: Basic CRUD
    def set(self, key: str, field: str, value: str) -> None:
        self.set_at_with_ttl(key, field, value, timestamp=0, ttl=None)

    def get(self, key: str, field: str) -> Optional[str]:
        return self.get_at(key, field, timestamp=0)

    def delete(self, key: str, field: str) -> bool:
        if key in self.db and field in self.db[key]:
            del self.db[key][field]
            return True
        return False

    # Level 2: Scanning & Output Formatting
    def scan(self, key: str) -> List[str]:
        return self.scan_at(key, timestamp=0)

    def scan_by_prefix(self, key: str, prefix: str) -> List[str]:
        return [entry for entry in self.scan(key) if entry.startswith(prefix)]

    # Level 3: Timestamps & Time-To-Live
    def set_at_with_ttl(self, key: str, field: str, value: str, timestamp: int, ttl: Optional[int]) -> None:
        self.db[key][field] = (value, timestamp, ttl)

    def get_at(self, key: str, field: str, timestamp: int) -> Optional[str]:
        if key in self.db and field in self.db[key]:
            val, set_t, ttl = self.db[key][field]
            if ttl is None or timestamp < set_t + ttl:
                return val
        return None

    def scan_at(self, key: str, timestamp: int) -> List[str]:
        if key not in self.db:
            return []
        res = []
        for f, (v, set_t, ttl) in self.db[key].items():
            if ttl is None or timestamp < set_t + ttl:
                res.append(f"{f}({v})")
        res.sort()
        return res

    # Level 4: Backup & Restore with Remaining TTL Recomputation
    def backup(self, timestamp: int) -> int:
        self.backup_counter += 1
        snapshot: Dict[str, Dict[str, Tuple[str, Optional[int]]]] = {}
        for k, fields in self.db.items():
            valid_fields = {}
            for f, (v, set_t, ttl) in fields.items():
                if ttl is None:
                    valid_fields[f] = (v, None)
                elif timestamp < set_t + ttl:
                    remaining_ttl = (set_t + ttl) - timestamp
                    valid_fields[f] = (v, remaining_ttl)
            if valid_fields:
                snapshot[k] = valid_fields
        self.backups[self.backup_counter] = snapshot
        return self.backup_counter

    def restore(self, timestamp: int, backup_id: int) -> bool:
        if backup_id not in self.backups:
            return False
        snapshot = self.backups[backup_id]
        self.db = defaultdict(dict)
        for k, fields in snapshot.items():
            for f, (v, rem_ttl) in fields.items():
                self.db[k][f] = (v, timestamp, rem_ttl)
        return True

# ---------------- Verification Tests ----------------
db = InMemoryDB()

# Level 1 & 2: CRUD and Scan
db.set("user1", "name", "Alice")
db.set("user1", "age", "30")
assert db.scan("user1") == ["age(30)", "name(Alice)"]

# Level 3: TTL expiration at set_time + ttl
db.set_at_with_ttl("user1", "session", "tok123", timestamp=10, ttl=5)
assert db.get_at("user1", "session", timestamp=12) == "tok123"
assert db.get_at("user1", "session", timestamp=15) is None, "Expired at 10+5=15"

# Level 4: Backup at t=12 (remaining TTL of session = 3)
bk = db.backup(timestamp=12)
db.delete("user1", "name")
assert db.get("user1", "name") is None

# Restore at t=20 -> session should expire at 20 + 3 = 23
assert db.restore(timestamp=20, backup_id=bk) is True
assert db.get_at("user1", "name", timestamp=20) == "Alice"
assert db.get_at("user1", "session", timestamp=22) == "tok123", "Valid before 20+3"
assert db.get_at("user1", "session", timestamp=24) is None, "Expired after 20+3"

print("✓ Problem 7: InMemoryDB 4-Level assessment passed.")


---
## 8. Cloud Storage Manager with User Quotas & Auto-Eviction

### Problem Statement
Design a multi-tenant cloud storage manager:
- **Level 1 (Files):** `add_file(name, size)`, `get_file_size(name)`, `copy_file(src, dest)`.
- **Level 2 (Search):** `find_by_prefix(prefix)`, `find_by_suffix(suffix)` returning lexicographically sorted filenames.
- **Level 3 (User Quotas & Eviction):** `add_user(user_id, capacity)`, `add_file_by_user(user_id, name, size)`.
  - If adding the file exceeds user capacity, the system **auto-evicts the user's largest files first** until enough space is available.
  - **Tie-breaker:** If multiple files have the same size, evict the **lexicographically larger name** first.
  - If the new file size alone exceeds the user's total capacity, reject the operation (`return False`).
- **Level 4 (Compression):** `compress_file(name)`:
  - Halves the file size (`size // 2`) and updates the owning user's consumed quota.
  - Returns `False` if file is already compressed.

### Starter Interface
```python
class CloudStorage:
    def add_file(self, name: str, size: int) -> bool: pass
    def get_file_size(self, name: str) -> int | None: pass
    def copy_file(self, src: str, dest: str) -> bool: pass
    def find_by_prefix(self, prefix: str) -> list[str]: pass
    def find_by_suffix(self, suffix: str) -> list[str]: pass
    def add_user(self, user_id: str, capacity: int) -> bool: pass
    def add_file_by_user(self, user_id: str, name: str, size: int) -> bool: pass
    def compress_file(self, name: str) -> bool: pass
```


In [ ]:
from __future__ import annotations
from typing import Dict, List, Optional, Set, Tuple

class CloudStorage:
    """Multi-tenant cloud storage with prefix/suffix search, auto-eviction, and compression."""
    def __init__(self):
        self.files: Dict[str, Tuple[int, Optional[str], bool]] = {}
        self.users: Dict[str, Dict[str, any]] = {}

    def add_file(self, name: str, size: int) -> bool:
        if name in self.files:
            return False
        self.files[name] = (size, None, False)
        return True

    def get_file_size(self, name: str) -> Optional[int]:
        return self.files[name][0] if name in self.files else None

    def copy_file(self, src: str, dest: str) -> bool:
        if src not in self.files or dest in self.files:
            return False
        size, user_id, comp = self.files[src]
        if user_id:
            return self.add_file_by_user(user_id, dest, size)
        self.files[dest] = (size, None, comp)
        return True

    def find_by_prefix(self, prefix: str) -> List[str]:
        return sorted([k for k in self.files if k.startswith(prefix)])

    def find_by_suffix(self, suffix: str) -> List[str]:
        return sorted([k for k in self.files if k.endswith(suffix)])

    def add_user(self, user_id: str, capacity: int) -> bool:
        if user_id in self.users:
            return False
        self.users[user_id] = {'capacity': capacity, 'used': 0, 'files': set()}
        return True

    def add_file_by_user(self, user_id: str, name: str, size: int) -> bool:
        if user_id not in self.users or name in self.files:
            return False
        user = self.users[user_id]
        if size > user['capacity']:
            return False

        # Evict largest files first, tie breaker: lexicographically larger name
        while user['used'] + size > user['capacity']:
            candidate = sorted(
                user['files'],
                key=lambda fname: (self.files[fname][0], fname),
                reverse=True
            )[0]
            cand_size = self.files[candidate][0]
            user['used'] -= cand_size
            user['files'].remove(candidate)
            del self.files[candidate]

        self.files[name] = (size, user_id, False)
        user['files'].add(name)
        user['used'] += size
        return True

    def compress_file(self, name: str) -> bool:
        if name not in self.files:
            return False
        size, user_id, comp = self.files[name]
        if comp:
            return False
        new_size = size // 2
        saved = size - new_size
        self.files[name] = (new_size, user_id, True)
        if user_id and user_id in self.users:
            self.users[user_id]['used'] -= saved
        return True

# ---------------- Verification Tests ----------------
cs = CloudStorage()
cs.add_user("alice", capacity=100)

assert cs.add_file_by_user("alice", "doc.pdf", 40) is True
assert cs.add_file_by_user("alice", "video.mp4", 50) is True
assert cs.users["alice"]["used"] == 90

# Adding 30 exceeds 100: should evict largest file ("video.mp4", 50)
assert cs.add_file_by_user("alice", "photo.png", 30) is True
assert "video.mp4" not in cs.files, "video.mp4 should have been auto-evicted"
assert cs.users["alice"]["used"] == 70

# Compression halves photo.png (30 -> 15)
assert cs.compress_file("photo.png") is True
assert cs.get_file_size("photo.png") == 15
assert cs.users["alice"]["used"] == 55

print("✓ Problem 8: CloudStorage auto-eviction and compression verified.")


---
## 9. Text Editor with Undo/Redo & Find/Replace

### Problem Statement
Implement a stateful text editor:
- **Level 1 (Typing & Cursor):**
  - `insert(text: str)`: Inserts string at the current cursor position and advances cursor.
  - `delete(k: int) -> int`: Deletes up to $k$ characters directly before the cursor.
  - `get_text() -> str`: Returns current text buffer.
- **Level 2 (Find & Replace):**
  - `find(pattern: str) -> list[int]`: Returns starting indices of all non-overlapping matches.
  - `replace_all(pattern: str, replacement: str) -> int`: Replaces all occurrences and returns the count of substitutions made.
- **Level 3 (Undo/Redo):**
  - `undo() -> bool` & `redo() -> bool`: Transactional command stack reverting/restoring buffer state. Any new modification clears the redo stack.

### Starter Interface
```python
class TextEditor:
    def insert(self, text: str) -> None: pass
    def delete(self, k: int) -> int: pass
    def get_text(self) -> str: pass
    def find(self, pattern: str) -> list[int]: pass
    def replace_all(self, pattern: str, replacement: str) -> int: pass
    def undo(self) -> bool: pass
    def redo(self) -> bool: pass
```


In [ ]:
from __future__ import annotations
from typing import List, Tuple

class TextEditor:
    """Text editor with cursor manipulation, pattern replacement, and dual-stack undo/redo."""
    def __init__(self):
        self.buffer: List[str] = []
        self.cursor: int = 0
        self.undo_stack: List[Tuple[List[str], int]] = []
        self.redo_stack: List[Tuple[List[str], int]] = []

    def _save_snapshot(self) -> None:
        self.undo_stack.append((list(self.buffer), self.cursor))
        self.redo_stack.clear()

    def insert(self, text: str) -> None:
        self._save_snapshot()
        self.buffer[self.cursor:self.cursor] = list(text)
        self.cursor += len(text)

    def delete(self, k: int) -> int:
        self._save_snapshot()
        actual_k = min(k, self.cursor)
        start = self.cursor - actual_k
        del self.buffer[start:self.cursor]
        self.cursor = start
        return actual_k

    def get_text(self) -> str:
        return "".join(self.buffer)

    def find(self, pattern: str) -> List[int]:
        text = self.get_text()
        indices = []
        idx = text.find(pattern)
        while idx != -1:
            indices.append(idx)
            idx = text.find(pattern, idx + len(pattern))
        return indices

    def replace_all(self, pattern: str, replacement: str) -> int:
        self._save_snapshot()
        text = self.get_text()
        count = text.count(pattern)
        if count > 0:
            new_text = text.replace(pattern, replacement)
            self.buffer = list(new_text)
            self.cursor = min(self.cursor, len(self.buffer))
        return count

    def undo(self) -> bool:
        if not self.undo_stack:
            return False
        self.redo_stack.append((list(self.buffer), self.cursor))
        self.buffer, self.cursor = self.undo_stack.pop()
        return True

    def redo(self) -> bool:
        if not self.redo_stack:
            return False
        self.undo_stack.append((list(self.buffer), self.cursor))
        self.buffer, self.cursor = self.redo_stack.pop()
        return True

# ---------------- Verification Tests ----------------
editor = TextEditor()
editor.insert("hello world")
assert editor.get_text() == "hello world"
assert editor.find("world") == [6]

# Replace and Undo
assert editor.replace_all("world", "python") == 1
assert editor.get_text() == "hello python"

assert editor.undo() is True
assert editor.get_text() == "hello world", "Undo should restore previous text"

assert editor.redo() is True
assert editor.get_text() == "hello python", "Redo should restore replaced text"

print("✓ Problem 9: TextEditor undo/redo verification passed.")


---
## 10. Multi-Account Financial Ledger with Atomic Rollbacks

### Problem Statement
Implement a double-entry financial ledger:
- `create_account(account_id: str, initial_balance: int) -> bool`
- `transfer(src: str, dest: str, amount: int, timestamp: int) -> str | None`:
  - Debits `src` and credits `dest` atomically.
  - Returns a unique transaction ID on success, or `None` if invalid/insufficient funds.
- `rollback(tx_id: str, timestamp: int) -> bool`:
  - Reverses the transaction atomically if `dest` currently has sufficient balance to refund the transfer.
- `get_balance_at(account_id: str, timestamp: int) -> int | None`:
  - Point-in-time balance query using historical logging and binary search ($\mathcal{O}(\log T)$).

### Starter Interface
```python
class TransactionLedger:
    def create_account(self, account_id: str, initial_balance: int) -> bool: pass
    def transfer(self, src: str, dest: str, amount: int, timestamp: int) -> str | None: pass
    def rollback(self, transaction_id: str, timestamp: int) -> bool: pass
    def get_balance_at(self, account_id: str, timestamp: int) -> int | None: pass
```


In [ ]:
from __future__ import annotations
import bisect
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

class TransactionLedger:
    """Double-entry transaction ledger with point-in-time binary search balance queries and atomic rollbacks."""
    def __init__(self):
        self.accounts: Dict[str, int] = {}
        self.balance_history: Dict[str, List[Tuple[int, int]]] = defaultdict(list)
        self.transactions: Dict[str, Dict[str, any]] = {}
        self.tx_counter = 0

    def create_account(self, account_id: str, initial_balance: int) -> bool:
        if account_id in self.accounts or initial_balance < 0:
            return False
        self.accounts[account_id] = initial_balance
        self.balance_history[account_id].append((0, initial_balance))
        return True

    def transfer(self, src: str, dest: str, amount: int, timestamp: int) -> Optional[str]:
        if src not in self.accounts or dest not in self.accounts:
            return None
        if amount <= 0 or self.accounts[src] < amount:
            return None

        self.accounts[src] -= amount
        self.accounts[dest] += amount
        self.balance_history[src].append((timestamp, self.accounts[src]))
        self.balance_history[dest].append((timestamp, self.accounts[dest]))

        self.tx_counter += 1
        tx_id = f"tx_{self.tx_counter}"
        self.transactions[tx_id] = {
            'src': src, 'dest': dest, 'amount': amount, 'status': 'COMMITTED'
        }
        return tx_id

    def rollback(self, tx_id: str, timestamp: int) -> bool:
        if tx_id not in self.transactions:
            return False
        tx = self.transactions[tx_id]
        if tx['status'] != 'COMMITTED':
            return False

        src, dest, amt = tx['src'], tx['dest'], tx['amount']
        if self.accounts[dest] < amt:
            return False

        self.accounts[dest] -= amt
        self.accounts[src] += amt
        self.balance_history[dest].append((timestamp, self.accounts[dest]))
        self.balance_history[src].append((timestamp, self.accounts[src]))
        tx['status'] = 'ROLLED_BACK'
        return True

    def get_balance_at(self, account_id: str, timestamp: int) -> Optional[int]:
        if account_id not in self.accounts:
            return None
        history = self.balance_history[account_id]
        idx = bisect.bisect_right(history, (timestamp, float('inf'))) - 1
        return history[idx][1] if idx >= 0 else 0

# ---------------- Verification Tests ----------------
ledger = TransactionLedger()
ledger.create_account("acc1", 100)
ledger.create_account("acc2", 50)

tx = ledger.transfer("acc1", "acc2", amount=30, timestamp=10)
assert tx is not None

assert ledger.get_balance_at("acc1", timestamp=5) == 100
assert ledger.get_balance_at("acc1", timestamp=10) == 70
assert ledger.get_balance_at("acc2", timestamp=10) == 80

assert ledger.rollback(tx, timestamp=15) is True
assert ledger.get_balance_at("acc1", timestamp=15) == 100
assert ledger.get_balance_at("acc2", timestamp=15) == 50

print("✓ Problem 10: TransactionLedger atomic rollback and history verified.")


---
## 11. Sliding Window Log & Token Bucket Rate Limiter

### Problem Statement
Implement two foundational rate-limiting algorithms:
1. **Sliding Window Log:**
   - Tracks exact request timestamps per client.
   - For an incoming request at `timestamp`, prunes timestamps older than `timestamp - window_seconds`.
   - Allows request if the active count $< max\_requests$.
2. **Token Bucket:**
   - Maintains a bucket with `capacity` and `refill_rate` (tokens/sec).
   - Dynamically computes refill: $tokens = \min(capacity, tokens + elapsed \times refill\_rate)$.
   - Allows request if $tokens \ge cost$.

### Starter Interface
```python
class SlidingWindowRateLimiter:
    def __init__(self, max_requests: int, window_seconds: float): pass
    def allow_request(self, client_id: str, timestamp: float) -> bool: pass

class TokenBucketRateLimiter:
    def __init__(self, capacity: float, refill_rate: float): pass
    def allow_request(self, client_id: str, timestamp: float, cost: float = 1.0) -> bool: pass
```


In [ ]:
from __future__ import annotations
from collections import defaultdict, deque
from typing import Dict, Tuple

class SlidingWindowRateLimiter:
    """Sliding window log rate limiter with microsecond timestamp pruning."""
    def __init__(self, max_requests: int, window_seconds: float):
        self.max_requests = max_requests
        self.window = window_seconds
        self.logs: Dict[str, deque[float]] = defaultdict(deque)

    def allow_request(self, client_id: str, timestamp: float) -> bool:
        q = self.logs[client_id]
        cutoff = timestamp - self.window
        while q and q[0] <= cutoff:
            q.popleft()
            
        if len(q) < self.max_requests:
            q.append(timestamp)
            return True
        return False

class TokenBucketRateLimiter:
    """Token bucket rate limiter supporting bursts with continuous fractional refill."""
    def __init__(self, capacity: float, refill_rate: float):
        self.capacity = capacity
        self.refill_rate = refill_rate
        self.buckets: Dict[str, Tuple[float, float]] = {}

    def allow_request(self, client_id: str, timestamp: float, cost: float = 1.0) -> bool:
        if client_id not in self.buckets:
            self.buckets[client_id] = (self.capacity, timestamp)

        tokens, last_time = self.buckets[client_id]
        elapsed = max(0.0, timestamp - last_time)
        tokens = min(self.capacity, tokens + elapsed * self.refill_rate)

        if tokens >= cost:
            self.buckets[client_id] = (tokens - cost, timestamp)
            return True
        else:
            self.buckets[client_id] = (tokens, timestamp)
            return False

# ---------------- Verification Tests ----------------
sw = SlidingWindowRateLimiter(max_requests=2, window_seconds=5.0)
assert sw.allow_request("c1", 1.0) is True
assert sw.allow_request("c1", 2.0) is True
assert sw.allow_request("c1", 3.0) is False, "Exceeded 2 requests in 5s window"
assert sw.allow_request("c1", 6.5) is True, "Timestamp 1.0 slid out of window [1.5, 6.5]"

tb = TokenBucketRateLimiter(capacity=3.0, refill_rate=1.0)
assert tb.allow_request("c1", 0.0) is True
assert tb.allow_request("c1", 0.0) is True
assert tb.allow_request("c1", 0.0) is True
assert tb.allow_request("c1", 0.0) is False
assert tb.allow_request("c1", 2.0) is True

print("✓ Problem 11: RateLimiters passed all window and token assertions.")


---
## 12. DAG Task Execution Engine with Dependency Resolution

### Problem Statement
Design an orchestration engine for Directed Acyclic Graph (DAG) task pipelines:
- `register_task(task_id: str, fn: callable, dependencies: list[str] = None)`:
  - Registers a task node and its parent dependency task IDs.
- `get_execution_order() -> list[str]`:
  - Computes a valid topological execution order using **Kahn's Algorithm**.
  - Raises `ValueError("Cycle detected in DAG task graph")` if a dependency cycle exists.
- `run() -> dict[str, any]`:
  - Executes the registered tasks in valid topological order.
  - Automatically feeds parent task outputs into child functions as a dictionary `{parent_id: parent_result}`.
  - Returns `{task_id: result}` dictionary.

### Starter Interface
```python
class DAGTaskEngine:
    def register_task(self, task_id: str, fn: callable, dependencies: list[str] = None) -> None: pass
    def get_execution_order(self) -> list[str]: pass
    def run(self) -> dict[str, any]: pass
```


In [ ]:
from __future__ import annotations
from collections import defaultdict, deque
from typing import Callable, Dict, List, Optional, Set, Any

class DAGTaskEngine:
    """Topological DAG task executor with circular dependency detection and output piping."""
    def __init__(self):
        self.tasks: Dict[str, Callable[[Dict[str, Any]], Any]] = {}
        self.deps: Dict[str, Set[str]] = {}
        self.dependents: Dict[str, Set[str]] = defaultdict(set)

    def register_task(self, task_id: str, fn: Callable[[Dict[str, Any]], Any], dependencies: Optional[List[str]] = None) -> None:
        self.tasks[task_id] = fn
        self.deps[task_id] = set(dependencies) if dependencies else set()
        for dep in self.deps[task_id]:
            self.dependents[dep].add(task_id)

    def get_execution_order(self) -> List[str]:
        in_degree = {t: len(self.deps[t]) for t in self.tasks}
        queue = deque([t for t, deg in in_degree.items() if deg == 0])
        order = []

        while queue:
            curr = queue.popleft()
            order.append(curr)
            for child in self.dependents[curr]:
                in_degree[child] -= 1
                if in_degree[child] == 0:
                    queue.append(child)

        if len(order) != len(self.tasks):
            raise ValueError("Cycle detected in DAG task graph")
        return order

    def run(self) -> Dict[str, Any]:
        order = self.get_execution_order()
        results: Dict[str, Any] = {}
        for task_id in order:
            parent_outputs = {p: results[p] for p in self.deps[task_id]}
            results[task_id] = self.tasks[task_id](parent_outputs)
        return results

# ---------------- Verification Tests ----------------
dag = DAGTaskEngine()
dag.register_task("fetch_features", lambda _: [1.0, 2.0, 3.0])
dag.register_task("fetch_weights", lambda _: [0.5, 0.5, 0.5])
dag.register_task(
    "compute_prediction",
    lambda parents: sum(f * w for f, w in zip(parents["fetch_features"], parents["fetch_weights"])),
    dependencies=["fetch_features", "fetch_weights"]
)
dag.register_task(
    "format_response",
    lambda parents: f"Prediction: {parents['compute_prediction']:.2f}",
    dependencies=["compute_prediction"]
)

order = dag.get_execution_order()
assert order.index("fetch_features") < order.index("compute_prediction")
assert order.index("fetch_weights") < order.index("compute_prediction")
assert order.index("compute_prediction") < order.index("format_response")

outputs = dag.run()
assert outputs["compute_prediction"] == 3.0
assert outputs["format_response"] == "Prediction: 3.00"

cyclic_dag = DAGTaskEngine()
cyclic_dag.register_task("X", lambda _: 1, dependencies=["Y"])
cyclic_dag.register_task("Y", lambda _: 2, dependencies=["X"])
try:
    cyclic_dag.get_execution_order()
    assert False, "Should have raised ValueError on cycle"
except ValueError as e:
    assert "Cycle detected" in str(e)

print("✓ Problem 12: DAGTaskEngine Kahn's topological sort and cycle detection verified.")
